# Работа с данными

Прочитаем и изучим train_identity.csv и train_transaction.csv - это таблицы, на которых будем обучать и проверять модель.

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
transact_path = "../data/hard/raw/train_transaction.csv"
ident_path = "../data/hard/raw/train_identity.csv"

transact_df = pd.read_csv(transact_path)
ident_df = pd.read_csv(ident_path)

Таблицы соединены через TransactionID. Одна таблица содержит информацию об устройстве, а другая о самой транзакции. Соединим таблицы по ID.
Также таблицы имеют колонку TransactionDT - она отвечает за время. То есть данные имеют временной характер, отсортируем по времени.
Также убедимся, что датасет очень несбалансированный.

In [3]:
df = transact_df.merge(ident_df, on='TransactionID', how='left')
df = df.sort_values('TransactionDT')

print(f'Процент мошеннических операций: {((sum(df['isFraud']) / len(df['isFraud'])) * 100):.2f}%')

Процент мошеннических операций: 3.50%


*Пропуски*: если в колонце больше 80% пропусков, то удаляем это колонку. Также удалим заведомо ненужные колонки.

In [4]:
missing = df.isnull().mean().sort_values(ascending=False)
missing = missing.drop(index=missing[missing == 0.0].index.tolist())
cols_to_drop = missing[missing > 0.8].index.tolist()

df = df.drop(columns=cols_to_drop)
df = df.drop(columns=['TransactionID', 'TransactionDT'])

missing = df.isnull().mean().sort_values(ascending=False)
missing = missing.drop(index=missing[missing == 0.0].index.tolist())

 Посмотрим названия колонок, где есть пропуски. Те, где много пропусков удалили из missing

In [5]:
print(missing.index.tolist())

['DeviceInfo', 'id_13', 'id_16', 'V275', 'V276', 'V277', 'V278', 'V273', 'V274', 'V262', 'V263', 'V265', 'V264', 'V266', 'V267', 'V261', 'V260', 'V223', 'V217', 'V241', 'V240', 'V243', 'V242', 'V235', 'V236', 'V230', 'V231', 'V229', 'V232', 'V224', 'V225', 'V226', 'V228', 'V249', 'V248', 'V218', 'V219', 'V237', 'V233', 'V246', 'V244', 'V254', 'V253', 'V247', 'V252', 'V257', 'V258', 'V268', 'V269', 'id_05', 'id_06', 'R_emaildomain', 'id_20', 'id_19', 'id_17', 'V186', 'V199', 'V213', 'V216', 'V191', 'V192', 'V190', 'V182', 'V183', 'V181', 'V204', 'V203', 'V202', 'V205', 'V207', 'V206', 'V211', 'V212', 'V179', 'V178', 'V177', 'V176', 'V173', 'V187', 'V215', 'V214', 'V168', 'V167', 'V172', 'V196', 'V193', 'V185', 'V194', 'V184', 'V201', 'V208', 'V210', 'V209', 'V200', 'V195', 'V188', 'V189', 'V171', 'V180', 'V170', 'V169', 'V175', 'V174', 'V198', 'V197', 'id_31', 'DeviceType', 'id_02', 'id_28', 'id_29', 'id_11', 'id_35', 'id_38', 'id_36', 'id_15', 'id_37', 'V251', 'V239', 'V250', 'V222', '

## Обработка пропусков

Необходимо обработать пропуски. Для начала узнаем, какие колонки - **категориальные**

In [6]:
cat_cols = df[missing.index].select_dtypes(include=['category', 'str']).columns
print(f'Категориальные колонки: {cat_cols.tolist()}')

Категориальные колонки: ['DeviceInfo', 'id_16', 'R_emaildomain', 'id_31', 'DeviceType', 'id_28', 'id_29', 'id_35', 'id_38', 'id_36', 'id_15', 'id_37', 'id_12', 'M5', 'M7', 'M9', 'M8', 'M4', 'M2', 'M3', 'M1', 'M6', 'P_emaildomain', 'card4', 'card6']


Также выберем колонки, которые являются числовыми, но с небольшим диапазоном значений.
Учитываем, что колонки начинающиеся на V - не категориальные, а числовые после некоторых преобразований.
Если взять порог в 50 уникальных экземпляров, то убеждаемся, что таких нет.
Значит это числовые признаки

In [7]:
num_cols = [col for col in missing.index.tolist() if (col not in cat_cols)]
num_cat_cols = []
for num_col in num_cols:
    if (df[num_col].nunique()) < 50 and not num_col.startswith('V'):
        num_cat_cols.append(num_col)
print(f'Числовые колонки с малым количеством уникальных значений:\n{num_cat_cols}')

Числовые колонки с малым количеством уникальных значений:
[]


Заполним пропуски. Все категориальные заполним просто "MISSING"

In [8]:
for col in cat_cols:
    df[col] = df[col].fillna("MISSING")

Убедимся, что в категориальных колонках больше нет пропусков и в целевой колонке тоже.

In [12]:
for col in cat_cols:
    print(f"{col}: {df[col].isnull().sum()} пропусков")
print(f"Пропусков в isFraud: {df['isFraud'].isnull().sum()}")

DeviceInfo: 0 пропусков
id_16: 0 пропусков
R_emaildomain: 0 пропусков
id_31: 0 пропусков
DeviceType: 0 пропусков
id_28: 0 пропусков
id_29: 0 пропусков
id_35: 0 пропусков
id_38: 0 пропусков
id_36: 0 пропусков
id_15: 0 пропусков
id_37: 0 пропусков
id_12: 0 пропусков
M5: 0 пропусков
M7: 0 пропусков
M9: 0 пропусков
M8: 0 пропусков
M4: 0 пропусков
M2: 0 пропусков
M3: 0 пропусков
M1: 0 пропусков
M6: 0 пропусков
P_emaildomain: 0 пропусков
card4: 0 пропусков
card6: 0 пропусков
Пропусков в isFraud: 0


EDA закончен!

## Обучение модели

Разделим данные на train/test в пропорции 0.8/0.2. Учитываем, что данные временные, поэтому train - первые 80% данных.

In [14]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

In [16]:
X_train = train_df.drop(columns=['isFraud'])
y_train = train_df['isFraud']
X_test = test_df.drop(columns=['isFraud'])
y_test = test_df['isFraud']

Категориальные признаки:

In [17]:
final_cat_cols = [col for col in cat_cols if col in X_train.columns]

Построим и обучим модель CatBoost. Числовые признаки не обрабатывали: CatBoost сам заполнит пропуски

In [ ]:
model = CatBoostClassifier(
    cat_features=final_cat_cols,
    auto_class_weights='Balanced',
    iterations=500,
    learning_rate=0.03,
    depth=6,
    verbose=100,
    random_seed=42
)

model.fit(X_train, y_train)